# Day 14: System Prompt Versioning System

## Core Theory (Just-in-Time)
As LLM applications move to production, prompts become critical infrastructure—similar to application code or database schemas. A slight modification to a system prompt can drastically alter model behavior, introduce regressions, or break downstream parsers.

### The "Why"
- **Reproducibility:** If an application starts failing, you need to know exactly which prompt was active at the time.
- **A/B Testing & Evaluation:** To compare prompt effectiveness, you must systematically track inputs, prompts, and corresponding outputs.
- **Rollbacks:** When a new prompt underperforms, you must be able to revert to a stable, known version instantly.

### The "How"
We implement prompt versioning by abstracting the prompt out of application code and into a managed registry. This can be achieved using a structured `PromptRegistry` class that relies on modern tooling like Pydantic for validation. Each prompt is treated as an immutable artifact with an explicit version.


## Code Implementation

We will create a `PromptRegistry` using Python, `pydantic` for data validation, and `langchain_core` for the prompt templates.

> Note: We use strict type hinting and docstrings as per our production-first requirements.


In [ ]:
import json
from typing import Dict, List, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

class PromptVersion(BaseModel):
    """Represents a single, immutable version of a system prompt."""
    version: str = Field(..., description="Semantic version string (e.g., '1.0.0')")
    template_str: str = Field(..., description="The raw prompt template string")
    description: str = Field(..., description="Reason for this version or changes made")

class PromptRegistry:
    """
    A registry to manage and version system prompts.
    
    This simulates a production registry that might typically be backed 
    by a database or a specialized platform like LangSmith or W&B.
    """
    def __init__(self) -> None:
        # Internal storage mapping prompt names to lists of their versions
        self._registry: Dict[str, List[PromptVersion]] = {}

    def register(self, name: str, version: str, template_str: str, description: str) -> None:
        """
        Registers a new version of a prompt.
        
        Args:
            name: The logical name of the prompt (e.g., 'customer_support').
            version: The semantic version string.
            template_str: The prompt template text.
            description: Context on what this prompt does or what changed.
        
        Raises:
            ValueError: If the version for this prompt name already exists.
        """
        if name not in self._registry:
            self._registry[name] = []
            
        # Check if version already exists
        for pv in self._registry[name]:
            if pv.version == version:
                raise ValueError(f"Prompt '{name}' version '{version}' already exists.")
                
        new_version = PromptVersion(
            version=version,
            template_str=template_str,
            description=description
        )
        self._registry[name].append(new_version)
        print(f"Registered {name} v{version}")

    def get_prompt(self, name: str, version: Optional[str] = None) -> PromptTemplate:
        """
        Retrieves a LangChain PromptTemplate for a specific version.
        If version is None, retrieves the latest version.
        
        Args:
            name: The logical name of the prompt.
            version: The specific version to retrieve, or None for latest.
            
        Returns:
            A LangChain PromptTemplate object.
            
        Raises:
            KeyError: If the prompt name is not found.
            ValueError: If the specified version is not found.
        """
        if name not in self._registry or not self._registry[name]:
            raise KeyError(f"Prompt '{name}' not found in registry.")
            
        versions = self._registry[name]
        
        if version is None:
            # Simple fallback to latest added (in production, you'd parse semver)
            target_version = versions[-1]
        else:
            target_version = next((pv for pv in versions if pv.version == version), None)
            if target_version is None:
                raise ValueError(f"Version '{version}' not found for prompt '{name}'.")
                
        return PromptTemplate.from_template(target_version.template_str)

# Example Usage
if __name__ == "__main__":
    registry = PromptRegistry()
    
    # Registering an initial prompt
    registry.register(
        name="support_agent",
        version="1.0.0",
        template_str="You are a helpful support agent. Answer the user's question: {question}",
        description="Initial support prompt"
    )
    
    # Registering an updated version to fix a hallucination issue
    registry.register(
        name="support_agent",
        version="1.1.0",
        template_str="You are a helpful support agent. Answer the user's question: {question}. If you do not know the answer, say 'I don't know'.",
        description="Added guardrail against hallucination"
    )
    
    # Fetching and formatting
    prompt_v1 = registry.get_prompt("support_agent", "1.0.0")
    print("\nv1.0.0 output:", prompt_v1.format(question="What is your return policy?"))
    
    prompt_v1_1 = registry.get_prompt("support_agent", "1.1.0")
    print("v1.1.0 output:", prompt_v1_1.format(question="What is your return policy?"))


## Common Pitfalls in Production

1.  **Hardcoding Prompts in Application Logic:** Embedding multi-line f-strings deep inside controller functions makes it impossible for non-engineers (like PMs or Prompt Engineers) to iterate.
2.  **Lack of Regression Testing:** Changing a prompt to fix a bug for one specific query often breaks the model's performance on 10 other queries. You must run a golden dataset against new prompt versions before deployment.
3.  **Missing Telemetry:** If you log model responses but do not log the *exact prompt version* that generated them, you cannot correlate user feedback (thumbs up/down) to specific prompt changes.


## Practical Lab / Homework

**Task:** Extend the `PromptRegistry` to include a lightweight "Evaluation Tracker". 
1. Create a method to log an execution of a prompt version, including the input variables, output text, and a success boolean.
2. Provide a fully working script executing this tracking.

**Constraint:** No pseudo-code. Use strict type hinting.


In [ ]:
import datetime

class EvaluationLog(BaseModel):
    """Tracks the performance of a specific prompt execution."""
    prompt_name: str
    version: str
    inputs: Dict[str, str]
    output: str
    success: bool
    timestamp: datetime.datetime = Field(default_factory=datetime.datetime.now)

class EvalTrackingPromptRegistry(PromptRegistry):
    """Extends the registry with evaluation tracking capabilities."""
    def __init__(self) -> None:
        super().__init__()
        self._evaluations: List[EvaluationLog] = []
        
    def log_evaluation(self, prompt_name: str, version: str, inputs: Dict[str, str], output: str, success: bool) -> None:
        """
        Logs the result of a prompt execution.
        """
        # Validate that the prompt exists
        self.get_prompt(prompt_name, version)
        
        log_entry = EvaluationLog(
            prompt_name=prompt_name,
            version=version,
            inputs=inputs,
            output=output,
            success=success
        )
        self._evaluations.append(log_entry)
        print(f"Logged evaluation for {prompt_name} v{version} (Success: {success})")
        
    def get_success_rate(self, prompt_name: str, version: str) -> float:
        """Calculates the success rate for a specific prompt version."""
        relevant_logs = [log for log in self._evaluations if log.prompt_name == prompt_name and log.version == version]
        if not relevant_logs:
            return 0.0
        successes = sum(1 for log in relevant_logs if log.success)
        return successes / len(relevant_logs)

# Lab Solution Execution
if __name__ == "__main__":
    eval_registry = EvalTrackingPromptRegistry()
    
    # Setup
    eval_registry.register(
        "sql_generator", 
        "1.0.0", 
        "Generate a SQL query for: {query}", 
        "Initial version"
    )
    
    # Simulate executions
    prompt = eval_registry.get_prompt("sql_generator", "1.0.0")
    
    # Execution 1 (Success)
    inputs1 = {"query": "Get all active users"}
    output1 = "SELECT * FROM users WHERE status = 'active';"
    eval_registry.log_evaluation("sql_generator", "1.0.0", inputs1, output1, success=True)
    
    # Execution 2 (Failure)
    inputs2 = {"query": "Drop the users table"}
    output2 = "DROP TABLE users;"
    eval_registry.log_evaluation("sql_generator", "1.0.0", inputs2, output2, success=False)
    
    # Check success rate
    rate = eval_registry.get_success_rate("sql_generator", "1.0.0")
    print(f"\nSuccess rate for sql_generator v1.0.0: {rate * 100}%")
